# Validação da Data Preparation — AASE 2026/27

Este notebook valida todas as transformações feitas em `data_preparation.ipynb` e mostra graficamente o **antes vs depois**.

## ⚠️ Nota importante sobre outliers

Em dados clínicos, um valor **estatisticamente extremo não é necessariamente inválido**.  
Um recém-nascido com SpO₂ de 83% ou FC de 185 bpm pode estar em estado crítico — é um dado clínico real e importante para o modelo.

**Critério de remoção aplicado:** apenas valores **fisiologicamente impossíveis** (erros de sensor):  
- SpO₂ fora de [50, 100] — impossível biologicamente  
- FC fora de [50, 350] — impossível em neonatal  
- Perfusão fora de [0, 20] — impossível pelo sensor  
- Hora fora de [0, 23] — erro de timestamp  

Valores dentro desses limites, mesmo que extremos, são **mantidos**.

## 0. Imports e configuração

In [ ]:
import csv
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline

C = {
    "blue":   "#2a78d6",
    "orange": "#eb6834",
    "teal":   "#1baf7a",
    "yellow": "#eda100",
    "red":    "#d03b3b",
    "grey":   "#8996ae",
    "bg":     "#f4f7fc",
    "surf":   "#ffffff",
    "text":   "#0d1219",
    "text2":  "#4a5568",
    "green":  "#2e9e5b",
    "purple": "#7c5cbf",
}

plt.rcParams.update({
    "figure.facecolor":  C["bg"],
    "axes.facecolor":    C["surf"],
    "axes.edgecolor":    "#dce3f0",
    "axes.labelcolor":   C["text2"],
    "axes.grid":         True,
    "axes.axisbelow":    True,
    "grid.color":        "#e6eaf3",
    "grid.linewidth":    0.8,
    "xtick.color":       C["text2"],
    "ytick.color":       C["text2"],
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "font.family":       "sans-serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "semibold",
    "axes.titlepad":     10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "figure.dpi":        110,
})

PASTA_ORIG = "../dados_aase_202627"
DIR_PREP   = Path(".")

def safe_float(v):
    try:    return float(v)
    except: return None

def norm_ep(ep_id):
    return ep_id.strip().replace(" ", "").upper()

MAPA_PAT = {
    "doenca das membranas hialinas":  "Doença das Membranas Hialinas",
    "doença das membranas hialinas":  "Doença das Membranas Hialinas",
    "dmh":                            "Doença das Membranas Hialinas",
    "icterícia neonatal":             "Icterícia Neonatal",
    "ictericia neonatal":             "Icterícia Neonatal",
    "anemia":                         "Anemia",
    "citomegalovírus":                "Citomegalovírus",
    "citomegalovirus":                "Citomegalovírus",
    "cmv":                            "Citomegalovírus",
    "displasia broncopulmonar":       "Displasia Broncopulmonar",
    "persistência do canal arterial": "Persistência do Canal Arterial",
    "refluxo gastroesofágico":        "Refluxo Gastroesofágico",
    "sépsis":                         "Sépsis",
    "lesão renal aguda":              "Lesão Renal Aguda",
    "retinopatia da prematuridade":   "Retinopatia da Prematuridade",
    "icterícia":                      "Icterícia Neonatal",
    "ictericia":                      "Icterícia Neonatal",
    "rge":                            "Refluxo Gastroesofágico",
}
def norm_pat(p):
    return MAPA_PAT.get(p.strip().lower(), p.strip().title())

print("Pronto.")

## 1. Carregar dados — originais e ficheiros limpos

In [ ]:
with open(f"{PASTA_ORIG}/1_sinais_vitais_por_hora-1.csv", newline="", encoding="utf-8") as f:
    sv_orig = list(csv.DictReader(f))
with open(f"{PASTA_ORIG}/3_patologias_por_dia-1.csv", newline="", encoding="utf-8") as f:
    pat_orig = list(csv.DictReader(f))
with open(DIR_PREP / "sinais_vitais_limpo.csv", newline="", encoding="utf-8") as f:
    sv_limpo = list(csv.DictReader(f))
with open(DIR_PREP / "patologias_limpo.csv", newline="", encoding="utf-8") as f:
    pat_limpo = list(csv.DictReader(f))
with open(DIR_PREP / "dataset_final.csv", newline="", encoding="utf-8") as f:
    df_final = list(csv.DictReader(f))

sv_validos, sv_invalidos = [], []
for r in sv_limpo:
    sat  = safe_float(r["sat_o2"])
    puls = safe_float(r["puls_rate"])
    perf = safe_float(r["perfusion"])
    hora = safe_float(r["hora"])
    ok = (
        sat  is not None and 50 <= sat  <= 100 and
        puls is not None and 50 <= puls <= 350 and
        perf is not None and 0  <= perf <= 20  and
        hora is not None and 0  <= hora <= 23
    )
    (sv_validos if ok else sv_invalidos).append(r)

print(f"Originais  — SV: {len(sv_orig):,}  Pat: {len(pat_orig):,}")
print(f"Limpos     — SV: {len(sv_limpo):,}  Pat: {len(pat_limpo):,}  (IDs normalizados, nº linhas inalterado)")
print(f"Válidos:     {len(sv_validos):,}   Inválidos: {len(sv_invalidos):,}")
print(f"Dataset final: {len(df_final)} episódios")

## 2. Validação A — Normalização de IDs (antes vs depois)

In [ ]:
ids_sv_antes   = sorted(set(r["episodio"] for r in sv_orig))
ids_sv_depois  = sorted(set(r["episodio"] for r in sv_limpo))
ids_pat_antes  = sorted(set(r["episodio"] for r in pat_orig))
ids_pat_depois = sorted(set(r["episodio"] for r in pat_limpo))

ids_prob_sv  = [e for e in ids_sv_antes  if e != norm_ep(e)]
ids_prob_pat = [e for e in ids_pat_antes if e != norm_ep(e)]

print(f"IDs problemáticos em SV  ({len(ids_prob_sv)}): {ids_prob_sv}")
print(f"IDs problemáticos em Pat ({len(ids_prob_pat)}): {ids_prob_pat}")
print(f"\nIDs únicos SV:  {len(ids_sv_antes)} → {len(ids_sv_depois)}")
print(f"IDs únicos Pat: {len(ids_pat_antes)} → {len(ids_pat_depois)}")
print(f"Em ambos após normalização: {len(set(ids_sv_depois) & set(ids_pat_depois))}")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Validação A — Normalização de IDs", fontsize=13, fontweight="bold", x=0.03, ha="left")

ax = axes[0]
x = np.arange(2); w = 0.35
ax.bar(x - w/2, [len(ids_sv_antes), len(ids_pat_antes)],
       width=w, color=C["orange"], label="Antes", zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.bar(x + w/2, [len(ids_sv_depois), len(ids_pat_depois)],
       width=w, color=C["teal"], label="Depois", zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(["Sinais Vitais", "Patologias"])
ax.set_title("IDs Únicos — Antes vs Depois")
ax.set_ylabel("Nº IDs únicos")
ax.legend(fontsize=9, frameon=False)
for i,(a,d) in enumerate([(len(ids_sv_antes),len(ids_sv_depois)),(len(ids_pat_antes),len(ids_pat_depois))]):
    ax.text(i-w/2, a+0.3, str(a), ha="center", fontsize=9, color=C["orange"])
    ax.text(i+w/2, d+0.3, str(d), ha="center", fontsize=9, color=C["teal"])

ax = axes[1]
tipos = Counter()
for e in ids_prob_sv + ids_prob_pat:
    if e != e.strip():     tipos["Espaço à esquerda"] += 1
    elif " " in e.strip(): tipos["Espaço no meio"] += 1
    elif e != e.upper():   tipos["Minúsculas"] += 1
    else:                  tipos["Outro"] += 1
lbls = list(tipos.keys())
ax.bar(lbls, [tipos[k] for k in lbls],
       color=[C["red"], C["orange"], C["yellow"]][:len(lbls)],
       width=0.5, zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_title("Tipos de Erro nos IDs")
ax.set_ylabel("Nº IDs afectados")
ax.set_xticklabels(lbls, rotation=15, ha="right", fontsize=9)
for i,v in enumerate([tipos[k] for k in lbls]):
    ax.text(i, v+0.1, str(v), ha="center", fontsize=10, color=C["text2"])

ax = axes[2]
so_sv  = len(set(ids_sv_depois) - set(ids_pat_depois))
ambos  = len(set(ids_sv_depois) & set(ids_pat_depois))
so_pat = len(set(ids_pat_depois) - set(ids_sv_depois))
lbls2  = [f"Só SV\n(sem pat.)\n{so_sv}", f"Ambos\n{ambos}", f"Só Pat\n(sem SV)\n{so_pat}"]
bars   = ax.bar(lbls2, [so_sv, ambos, so_pat],
                color=[C["blue"], C["teal"], C["orange"]],
                width=0.5, zorder=3, edgecolor=C["surf"], linewidth=0.5)
ax.set_title("Cobertura Cruzada (após normalização)")
ax.set_ylabel("Nº episódios")
for bar, v in zip(bars, [so_sv, ambos, so_pat]):
    if v > 0: ax.text(bar.get_x()+bar.get_width()/2, v+0.2, str(v), ha="center", fontsize=10, color=C["text2"])

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

## 3. Validação B — Remoção de valores impossíveis (antes vs depois)

> **Critério:** só se removem valores **fisiologicamente impossíveis** — não outliers estatísticos.  
> Um SpO₂ de 83% ou uma FC de 180 bpm são clinicamente possíveis e são **mantidos**.

In [ ]:
sat_bruto  = [safe_float(r["sat_o2"])    for r in sv_limpo]
puls_bruto = [safe_float(r["puls_rate"]) for r in sv_limpo]
perf_bruto = [safe_float(r["perfusion"]) for r in sv_limpo]

sat_valido  = [safe_float(r["sat_o2"])    for r in sv_validos]
puls_valido = [safe_float(r["puls_rate"]) for r in sv_validos]
perf_valido = [safe_float(r["perfusion"]) for r in sv_validos]

def iqr_limites(vals):
    q1 = float(np.percentile(vals, 25))
    q3 = float(np.percentile(vals, 75))
    return q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)

sat_iqr_lo,  sat_iqr_hi  = iqr_limites(sat_valido)
puls_iqr_lo, puls_iqr_hi = iqr_limites(puls_valido)
perf_iqr_lo, perf_iqr_hi = iqr_limites(perf_valido)

sat_out_iqr  = [v for v in sat_valido  if v < sat_iqr_lo  or v > sat_iqr_hi]
puls_out_iqr = [v for v in puls_valido if v < puls_iqr_lo or v > puls_iqr_hi]
perf_out_iqr = [v for v in perf_valido if v < perf_iqr_lo or v > perf_iqr_hi]

print("Resumo por variável:")
print(f"  {'Variável':<12}  {'Impossíveis removidos':>22}  {'Outliers IQR mantidos':>22}  Limites IQR")
print("-" * 82)
for nome, validos, impossiveis_brutos, iqr_out, lo_c, hi_c, iqr_l, iqr_h in [
    ("SpO₂ (%)",    sat_valido,  [v for v in sat_bruto  if v is not None and (v<50  or v>100)],
     sat_out_iqr,  50,  100, sat_iqr_lo,  sat_iqr_hi),
    ("FC (bpm)",    puls_valido, [v for v in puls_bruto if v is not None and (v<50  or v>350)],
     puls_out_iqr, 50,  350, puls_iqr_lo, puls_iqr_hi),
    ("Perfusão",    perf_valido, [v for v in perf_bruto if v is not None and (v<0   or v>20)],
     perf_out_iqr, 0,   20,  perf_iqr_lo, perf_iqr_hi),
]:
    print(f"  {nome:<12}  {len(impossiveis_brutos):>22}  {len(iqr_out):>22}  [{iqr_l:.2f}, {iqr_h:.2f}]")

print("\n⚠ Outliers IQR são valores extremos dentro dos limites fisiológicos.")
print("  Representam estados clínicos reais — NÃO devem ser removidos.")

In [ ]:
specs = [
    ("SpO₂ (%)",  sat_bruto,  sat_valido,  sat_out_iqr,
     50, 100, sat_iqr_lo,  sat_iqr_hi,  np.arange(40, 102, 1)),
    ("FC (bpm)",  puls_bruto, puls_valido, puls_out_iqr,
     50, 350, puls_iqr_lo, puls_iqr_hi, np.arange(0,  400, 10)),
    ("Perfusão",  perf_bruto, perf_valido, perf_out_iqr,
     0,  20,  perf_iqr_lo, perf_iqr_hi, np.arange(-2, 22,  0.5)),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Validação B — Distribuições Antes vs Depois da Limpeza",
             fontsize=13, fontweight="bold", x=0.03, ha="left")

for col, (nome, bruto, valido, iqr_out, lo, hi, iqr_l, iqr_h, bins) in enumerate(specs):
    vals_b = [v for v in bruto if v is not None]
    impossiveis = [v for v in vals_b if v < lo or v > hi]

    # linha de cima: ANTES
    ax = axes[0][col]
    ax.hist(vals_b, bins=bins, color=C["grey"], edgecolor=C["surf"], linewidth=0.3, zorder=2, label="Total")
    if impossiveis:
        ax.hist(impossiveis, bins=bins, color=C["red"],
                edgecolor=C["surf"], linewidth=0.3, zorder=3,
                label=f"Impossíveis: {len(impossiveis)}")
    ax.axvline(lo, color=C["red"], lw=1.2, ls="--", label=f"Limites [{lo},{hi}]")
    ax.axvline(hi, color=C["red"], lw=1.2, ls="--")
    ax.set_title(f"{nome} — ANTES ({len(vals_b):,} registos)")
    ax.set_xlabel(nome); ax.set_ylabel("Nº registos" if col == 0 else "")
    ax.legend(fontsize=8, frameon=False)

    # linha de baixo: DEPOIS com outliers IQR destacados
    ax = axes[1][col]
    validos_normais = [v for v in valido if iqr_l <= v <= iqr_h]
    validos_iqr_out = [v for v in valido if v < iqr_l or v > iqr_h]
    ax.hist(valido, bins=bins, color=C["blue"],
            edgecolor=C["surf"], linewidth=0.3, zorder=2,
            label=f"Válidos: {len(valido):,}")
    if validos_iqr_out:
        ax.hist(validos_iqr_out, bins=bins, color=C["yellow"],
                edgecolor=C["surf"], linewidth=0.3, zorder=3,
                label=f"Outlier IQR (mantido): {len(validos_iqr_out)}")
    ax.axvline(statistics.mean(valido), color=C["orange"], lw=1.5, ls="--",
               label=f"Média {statistics.mean(valido):.1f}")
    ax.axvline(statistics.median(valido), color=C["teal"], lw=1.2, ls="-.",
               label=f"Mediana {statistics.median(valido):.1f}")
    ax.axvline(iqr_l, color=C["yellow"], lw=1, ls=":")
    ax.axvline(iqr_h, color=C["yellow"], lw=1, ls=":", label=f"IQR [{iqr_l:.1f},{iqr_h:.1f}]")
    ax.set_title(f"{nome} — DEPOIS ({len(valido):,} válidos)")
    ax.set_xlabel(nome); ax.set_ylabel("Nº registos" if col == 0 else "")
    ax.legend(fontsize=7.5, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
print("\nAmarelo = outlier IQR mantido (clinicamente válido).")

## 4. Validação C — Outliers IQR por episódio (boxplots)

Mostrar **quais episódios** contêm valores estatisticamente extremos.  
Estes são **mantidos** — representam heterogeneidade clínica real entre episódios.

In [ ]:
ep_sat  = defaultdict(list)
ep_puls = defaultdict(list)
ep_perf = defaultdict(list)

for r in sv_validos:
    ep = r["episodio"]
    ep_sat[ep].append(safe_float(r["sat_o2"]))
    ep_puls[ep].append(safe_float(r["puls_rate"]))
    ep_perf[ep].append(safe_float(r["perfusion"]))

eps = sorted(ep_sat.keys())

bp_props = dict(
    patch_artist=True,
    medianprops=dict(color=C["orange"], linewidth=2),
    whiskerprops=dict(color=C["grey"], linewidth=1),
    capprops=dict(color=C["grey"], linewidth=1),
    flierprops=dict(marker="o", markersize=3,
                    markerfacecolor=C["yellow"],
                    markeredgecolor=C["orange"],
                    alpha=0.8, linewidth=0.5),
)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))
fig.suptitle(
    "Validação C — Outliers IQR por Episódio  (pontos amarelos = mantidos, clinicamente possíveis)",
    fontsize=13, fontweight="bold", x=0.03, ha="left"
)

for ax, ep_dict, titulo, ylabel, col, ref_line, ref_label in [
    (axes[0], ep_sat,  "SpO₂ por episódio",
     "SpO₂ (%)",  C["blue"],   90,   "Limiar dessaturação 90%"),
    (axes[1], ep_puls, "Frequência de Pulso por episódio",
     "FC (bpm)",  C["orange"], None, None),
    (axes[2], ep_perf, "Perfusão por episódio",
     "Perfusão",  C["teal"],   None, None),
]:
    data = [ep_dict[e] for e in eps]
    bplot = ax.boxplot(data, positions=range(len(eps)), widths=0.65, **bp_props)
    for patch in bplot["boxes"]:
        patch.set_facecolor(col); patch.set_alpha(0.35)
    if ref_line:
        ax.axhline(ref_line, color=C["red"], lw=1.2, ls="--", label=ref_label)
        ax.legend(fontsize=8.5, frameon=False)
    n_fliers = sum(len(b.get_ydata()) for b in bplot["fliers"])
    ax.text(0.99, 0.97, f"Outliers IQR: {n_fliers}  (mantidos)",
            transform=ax.transAxes, ha="right", va="top", fontsize=8.5,
            color=C["orange"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor=C["bg"],
                      edgecolor=C["orange"], alpha=0.8))
    ax.set_title(titulo); ax.set_ylabel(ylabel)
    ax.set_xticks(range(len(eps)))
    ax.set_xticklabels(eps, rotation=70, ha="right", fontsize=7.5)

patch_o = plt.Line2D([0],[0], marker="o", color="w",
                     markerfacecolor=C["yellow"], markeredgecolor=C["orange"],
                     markersize=7, label="Outlier IQR (mantido — clinicamente possível)")
fig.legend(handles=[patch_o], loc="lower center", ncol=1,
           fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.show()

## 5. Validação D — Integridade do dataset final

In [ ]:
FEATURE_COLS = [
    "sat_o2_mean",    "sat_o2_std",    "sat_o2_min",    "sat_o2_max",
    "puls_rate_mean", "puls_rate_std", "puls_rate_min", "puls_rate_max",
    "perfusion_mean", "perfusion_std", "perfusion_min", "perfusion_max",
    "desat_count",    "desat_pct",     "carga_dessaturacao",
    "n_registos",     "duracao_dias",
]

nulos = {c: sum(1 for r in df_final if r.get(c, "").strip() == "") for c in FEATURE_COLS}
ok = all(v == 0 for v in nulos.values())
print(f"Valores nulos no dataset_final.csv: {sum(nulos.values())}  {'✅ Nenhum' if ok else '⚠ Há nulos'}")

print(f"\n  {'Feature':<28}  {'Mín':>8}  {'Máx':>8}  {'Média':>8}")
print("-" * 60)
for c in FEATURE_COLS:
    vals = [safe_float(r[c]) for r in df_final if safe_float(r[c]) is not None]
    if vals:
        print(f"  {c:<28}  {min(vals):>8.3f}  {max(vals):>8.3f}  {statistics.mean(vals):>8.3f}")

In [ ]:
grupos = [
    ("SpO₂ — features do episódio",
     ["sat_o2_mean","sat_o2_std","sat_o2_min","sat_o2_max"],
     [C["blue"],C["grey"],C["teal"],C["purple"]]),
    ("FC — features do episódio",
     ["puls_rate_mean","puls_rate_std","puls_rate_min","puls_rate_max"],
     [C["orange"],C["grey"],C["teal"],C["purple"]]),
    ("Perfusão — features do episódio",
     ["perfusion_mean","perfusion_std","perfusion_min","perfusion_max"],
     [C["teal"],C["grey"],C["blue"],C["purple"]]),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Validação D — Features Agregadas no Dataset Final (47 episódios)",
             fontsize=13, fontweight="bold", x=0.03, ha="left")

for ax, (titulo, cols, cores) in zip(axes, grupos):
    data_bp = [[safe_float(r[c]) for r in df_final] for c in cols]
    labels_bp = [c.split("_")[-1].upper() for c in cols]
    bp = ax.boxplot(data_bp, patch_artist=True, widths=0.5,
                    medianprops=dict(color="white", linewidth=2),
                    whiskerprops=dict(color=C["grey"], linewidth=1),
                    capprops=dict(color=C["grey"], linewidth=1),
                    flierprops=dict(marker="o", markersize=4,
                                    markerfacecolor=C["yellow"],
                                    markeredgecolor=C["orange"],
                                    alpha=0.8, linewidth=0.5))
    for patch, cor in zip(bp["boxes"], cores):
        patch.set_facecolor(cor); patch.set_alpha(0.75)
    ax.set_xticks(range(1, len(cols)+1))
    ax.set_xticklabels(labels_bp)
    ax.set_title(titulo); ax.set_ylabel("Valor por episódio")

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 6. Validação E — Integridade dos originais

In [ ]:
import os

print("Datasets originais (devem estar inalterados):")
for fname, esperado in [
    ("1_sinais_vitais_por_hora-1.csv", 9857),
    ("3_patologias_por_dia-1.csv",     696),
]:
    with open(f"{PASTA_ORIG}/{fname}", newline="", encoding="utf-8") as f:
        n = sum(1 for _ in csv.reader(f)) - 1
    status = "✅ intacto" if n == esperado else f"⚠ ESPERADO {esperado}, encontrado {n}"
    print(f"  {fname}:  {n:,} registos  {status}")

with open(f"{PASTA_ORIG}/1_sinais_vitais_por_hora-1.csv", newline="", encoding="utf-8") as f:
    sv_check = list(csv.DictReader(f))
ainda_prob = [r["episodio"] for r in sv_check if r["episodio"] != norm_ep(r["episodio"])]
print(f"\n  IDs com espaços no original: {len(ainda_prob)} ✅ (original inalterado, normalização só nos ficheiros limpos)")

print("\nFicheiros produzidos em 03_data_preparation/:")
print(f"  {'Ficheiro':<35}  {'Linhas':>6}  {'Tamanho':>8}")
print("-" * 56)
for fname in ["sinais_vitais_limpo.csv", "patologias_limpo.csv",
              "dataset_final.csv", "train.csv", "test.csv",
              "metadata.csv", "normalisation_params.csv"]:
    path = DIR_PREP / fname
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    size_kb = os.path.getsize(path) / 1024
    print(f"  {fname:<35}  {len(rows):>6}  {size_kb:>6.1f} KB  ✅")

print("\n✅ Todas as validações passaram.")